In [1]:
# Imports
import math
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
from scipy import stats
import seaborn as sns
from collections import defaultdict


#Getting the Average success values from the Rag and No Rag runs
df_cheap_norag1 = pd.read_csv('final_data/run1/vitis_oneshot_cheap_models_norag.csv')
df_cheap_rag1 = pd.read_csv('final_data/run1/vitis_oneshot_cheap_models_rag.csv')
df_cheap_norag2 = pd.read_csv('final_data/run2/vitis_oneshot_cheap_models_norag.csv')
df_cheap_rag2 = pd.read_csv('final_data/run2/vitis_oneshot_cheap_models_rag.csv')
df_expensive_norag1 = pd.read_csv('final_data/run1/vitis_oneshot_expensive_models_norag.csv')
df_expensive_rag1 = pd.read_csv('final_data/run1/vitis_oneshot_expensive_models_rag.csv')
df_expensive_norag2 = pd.read_csv('final_data/run2/vitis_oneshot_expensive_models_norag.csv')
df_expensive_rag2 = pd.read_csv('final_data/run2/vitis_oneshot_expensive_models_rag.csv')

# Cobining the different cheap and expensive dfs
df_rag1 = pd.concat([df_cheap_rag1, df_expensive_rag1], ignore_index=True)
df_rag2 = pd.concat([df_cheap_rag2, df_expensive_rag2], ignore_index=True)
df_norag1 = pd.concat([df_cheap_norag1, df_expensive_norag1], ignore_index=True)
df_norag2 = pd.concat([df_cheap_norag2, df_expensive_norag2], ignore_index=True)

# Formatting the successes and model names
df_rag1['latency'] = pd.to_numeric(df_rag1['latency'], errors='coerce')
df_rag1['Success'] = ((df_rag1['latency'].notna()) & (df_rag1['latency'] > 5)).astype(int)
df_rag1['model'] = df_rag1['model'].str.split('/').str[-1]

df_rag2['latency'] = pd.to_numeric(df_rag2['latency'], errors='coerce')
df_rag2['Success'] = ((df_rag2['latency'].notna()) & (df_rag2['latency'] > 5)).astype(int)
df_rag2['model'] = df_rag2['model'].str.split('/').str[-1]

df_norag1['latency'] = pd.to_numeric(df_norag1['latency'], errors='coerce')
df_norag1['Success'] = ((df_norag1['latency'].notna()) & (df_norag1['latency'] > 5)).astype(int)
df_norag1['model'] = df_norag1['model'].str.split('/').str[-1]

df_norag2['latency'] = pd.to_numeric(df_norag2['latency'], errors='coerce')
df_norag2['Success'] = ((df_norag2['latency'].notna()) & (df_norag2['latency'] > 5)).astype(int)
df_norag2['model'] = df_norag2['model'].str.split('/').str[-1]

#Getting success Rates
success_cnts_unsort = df_rag1.groupby('model')['Success'].sum()
success_counts = df_rag1.groupby('model')['Success'].sum().sort_values(ascending=False)
total_tests = 101
success_rag1 = (success_counts / total_tests) * 100
success_rag1_unsort = (success_cnts_unsort / total_tests) * 100

success_cnts_unsort = df_rag2.groupby('model')['Success'].sum()
success_counts = df_rag2.groupby('model')['Success'].sum().sort_values(ascending=False)
total_tests = 101
success_rag2 = (success_counts / total_tests) * 100
success_rag2_unsort = (success_cnts_unsort / total_tests) * 100

success_cnts_unsort = df_norag1.groupby('model')['Success'].sum()
success_counts = df_norag1.groupby('model')['Success'].sum().sort_values(ascending=False)
total_tests = 101
success_norag1 = (success_counts / total_tests) * 100
success_norag1_unsort = (success_cnts_unsort / total_tests) * 100

success_cnts_unsort = df_norag2.groupby('model')['Success'].sum()
success_counts = df_norag2.groupby('model')['Success'].sum().sort_values(ascending=False)
total_tests = 101
success_norag2 = (success_counts / total_tests) * 100
success_norag2_unsort = (success_cnts_unsort / total_tests) * 100

#Averaging the success rates
avg_success_rag = (success_rag1 + success_rag2) / 2
avg_success_norag = (success_norag1 + success_norag2) / 2
# print(success_rag1)
# print(success_rag2)
# print(avg_success_rag)
# print(avg_success_norag)

#Get Average Price per Call in each Rag v NoRag run
model_costs_data_norag = {
    'Model': ['gemini-2.5-flash', 'deepseek-chat-v3-0324', 'deepseek-r1-0528', 'gpt-4.1-mini', 'grok-code-fast-1',  
              'qwen3-coder', 'gpt-5', 'grok-4', 'gemini-2.5-pro', 'claude-sonnet-4.5'],
    'Base_Cost_Per_Run': [0.11, 0.04, 0.84, 0.06, 0.13, 0.03, 2.44, 5.26, 3.83, 0.81]
}
costs_norag = pd.DataFrame(model_costs_data_norag).set_index('Model')
success_by_costs_norag = pd.DataFrame({
    'Success_Rate': model_costs_data_norag,
    'Cost': costs_norag['Base_Cost_Per_Run'] # Assuming this column name in your data
}).dropna() # Drop any model that might be missing one of the values

model_costs_data_rag = {
    'Model': ['gemini-2.5-flash', 'deepseek-chat-v3-0324', 'deepseek-r1-0528', 'gpt-4.1-mini', 'grok-code-fast-1',  
              'qwen3-coder', 'gpt-5', 'grok-4', 'gemini-2.5-pro', 'claude-sonnet-4.5'],
    'Base_Cost_Per_Run': [0.22, 0.12, 0.92, 0.17, 0.18, 0.09, 2.93, 7.90, 6.13, 1.92]
}
costs_rag = pd.DataFrame(model_costs_data_rag).set_index('Model')
success_by_costs_rag = pd.DataFrame({
    'Success_Rate': model_costs_data_rag,
    'Cost': costs_rag['Base_Cost_Per_Run'] # Assuming this column name in your data
}).dropna() # Drop any model that might be missing one of the values

In [11]:
def find_never_passing_kernels(df):
    """
    Find kernels that never have both status='passed' AND match='match'.
    
    Parameters:
    - df: pandas DataFrame with columns: model, kernel, status, match
    
    Returns:
    - Dictionary with model names as keys and lists of never-passing kernels as values
    """
    # Group by model and kernel, check if any run passes both conditions
    grouped = df.groupby(['model', 'kernel'], group_keys=False)[['status', 'match']].apply(
        lambda x: ((x['status'] == 'passed') & (x['match'] == 'match')).any(),
        include_groups=False
    ).reset_index(name='has_success')
    
    # Filter for kernels that never succeed
    never_passing = grouped[~grouped['has_success']]
    
    # Organize by model
    result = defaultdict(list)
    for _, row in never_passing.iterrows():
        result[row['model']].append(row['kernel'])
    
    return dict(result)


def print_never_passing_summary(never_passing_dict):
    """
    Print a nicely formatted summary of never-passing kernels by model.
    
    Parameters:
    - never_passing_dict: output from find_never_passing_kernels()
    """
    print("=" * 60)
    print("KERNELS THAT NEVER PASS (by Model)")
    print("=" * 60)
    
    if not never_passing_dict:
        print("All kernels have at least one successful run!")
        return
    
    for model, kernels in sorted(never_passing_dict.items()):
        print(f"\n{model}: ({len(kernels)} kernels)")
        print("-" * 60)
        for kernel in sorted(kernels):
            print(f"  • {kernel}")
    
    print("\n" + "=" * 60)
    total = sum(len(kernels) for kernels in never_passing_dict.values())
    print(f"Total never-passing kernels: {total}")
    print("=" * 60)


def get_intersection_kernels(dict1, dict2):
    """
    Get only the intersection (AND) - kernels that fail in BOTH runs.
    
    Parameters:
    - dict1, dict2: dictionaries from find_never_passing_kernels()
    
    Returns:
    - Dictionary with model names as keys and lists of kernels that fail in both
    """
    result = {}
    all_models = set(dict1.keys()) | set(dict2.keys())
    
    for model in all_models:
        kernels1 = set(dict1.get(model, []))
        kernels2 = set(dict2.get(model, []))
        
        # Intersection: kernels in both
        common_kernels = sorted(list(kernels1 & kernels2))
        
        if common_kernels:
            result[model] = common_kernels
    
    return result

def print_intersection_summary(intersection_dict, name1="First", name2="Second"):
    """
    Print a summary of intersection kernels (those that fail in BOTH runs).
    """
    print("=" * 70)
    print(f"INTERSECTION ANALYSIS: Kernels that fail in BOTH runs")
    print(f"({name1} AND {name2})")
    print("=" * 70)
    
    if not intersection_dict:
        print("\nNo common failures - no kernels fail in both runs!")
        print("=" * 70)
        return
    
    total_common = 0
    for model in sorted(intersection_dict.keys()):
        kernels = intersection_dict[model]
        total_common += len(kernels)
        print(f"\n{model}: ({len(kernels)} common failures)")
        print("-" * 70)
        for kernel in kernels:
            print(f"  • {kernel}")
    
    print("\n" + "=" * 70)
    print(f"Total kernels that fail in both: {total_common}")
    print("=" * 70)

def find_kernels_failing_all_models(never_passing_dicts):
    """
    Find kernels that fail across ALL models in all provided dictionaries.
    
    Parameters:
    - never_passing_dicts: list of dictionaries from find_never_passing_kernels()
    
    Returns:
    - List of kernel names that fail in every model across all runs
    """
    if not never_passing_dicts:
        return []
    
    # Collect all models and all kernels
    all_models = set()
    kernel_failures = defaultdict(set)  # kernel -> set of models it fails in
    
    for d in never_passing_dicts:
        all_models.update(d.keys())
        for model, kernels in d.items():
            for kernel in kernels:
                kernel_failures[kernel].add(model)
    
    # Find kernels that fail in ALL models
    num_models = len(all_models)
    failing_all = [kernel for kernel, models in kernel_failures.items() 
                   if len(models) == num_models]
    
    return sorted(failing_all)

def print_failing_all_models_summary(failing_kernels, dict_names=None):
    """
    Print a summary of kernels that fail across all models.
    
    Parameters:
    - failing_kernels: list from find_kernels_failing_all_models()
    - dict_names: optional list of dictionary names for context
    """
    print("=" * 70)
    print("KERNELS FAILING ACROSS ALL MODELS")
    if dict_names:
        print(f"Datasets analyzed: {', '.join(dict_names)}")
    print("=" * 70)
    
    if not failing_kernels:
        print("\nNo kernels fail across all models!")
        print("=" * 70)
        return
    
    print(f"\nTotal kernels failing in ALL models: {len(failing_kernels)}")
    print("-" * 70)
    for kernel in failing_kernels:
        print(f"  • {kernel}")
    
    print("\n" + "=" * 70)

In [28]:
never_passing_norag1 = find_never_passing_kernels(df_norag1)
never_passing_norag2 = find_never_passing_kernels(df_norag2)
fail_always_norag = get_intersection_kernels(never_passing_norag2, never_passing_norag1)
never_passing_rag1 = find_never_passing_kernels(df_rag1)
never_passing_rag2 = find_never_passing_kernels(df_rag2)
fail_always_rag = get_intersection_kernels(never_passing_rag1, never_passing_rag2)
# print_never_passing_summary(never_passing_norag2)

print_failing_all_models_summary(find_kernels_failing_all_models([never_passing_norag1, never_passing_norag2, never_passing_rag1, never_passing_rag2]))
# print_intersection_summary(fail_always_norag)

print(df_norag2[df_norag2['kernel'] == 'round_bfloat16'])


KERNELS FAILING ACROSS ALL MODELS

Total kernels failing in ALL models: 6
----------------------------------------------------------------------
  • cast_bfloat16_to_int8
  • cumsum_bfloat16
  • floor_bfloat16
  • inverse_uint8
  • mse_loss_bfloat16
  • round_bfloat16

                     model  iteration          kernel  attempts  \
48            gpt-4.1-mini          1  round_bfloat16         1   
149            qwen3-coder          1  round_bfloat16         1   
250       gemini-2.5-flash          1  round_bfloat16         1   
351  deepseek-chat-v3-0324          1  round_bfloat16         1   
452       deepseek-r1-0528          1  round_bfloat16         1   
553       grok-code-fast-1          1  round_bfloat16         1   
654      claude-sonnet-4.5          1  round_bfloat16         1   
755                  gpt-5          1  round_bfloat16         1   
856                 grok-4          1  round_bfloat16         1   
957         gemini-2.5-pro          1  round_bfloat16       